In [ ]:
!pip install sklearn

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

df = iris.frame

df.head()

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 3, Finished, Available, Finished, False)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [1]:
import sempy.fabric as fabric

StatementMeta(, d4d0a883-7624-496e-9a20-8dc67f3f31f5, 3, Finished, Available, Finished, False)

In [2]:
workspaces = fabric.list_workspaces()
display(workspaces)

StatementMeta(, d4d0a883-7624-496e-9a20-8dc67f3f31f5, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ac509c4-c0a4-49a7-8a2f-bb8012f0e92f)

In [4]:
fabric.list_datasets()

StatementMeta(, d4d0a883-7624-496e-9a20-8dc67f3f31f5, 7, Finished, Available, Finished, False)

,Dataset Name,Dataset ID,Created Timestamp,Last Update
0,tavantsemantic,4670e92e-cd8f-4f0e-ac19-4ca456a73eaf,2025-10-21 18:57:49,NaT


In [ ]:
df_1=fabric.read_table("factsales","")

In [2]:
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 5, Finished, Available, Finished, False)

In [3]:
mlflow.autolog()

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 6, Finished, Available, Finished, False)

2026/06/08 00:14:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [4]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 7, Finished, Available, Finished, False)

In [5]:
experiment_name = "Fabric_MLflow_Demo"

mlflow.set_experiment(experiment_name)

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 8, Finished, Available, Finished, False)

2026/06/08 00:15:01 INFO mlflow.tracking.fluent: Experiment with name 'Fabric_MLflow_Demo' does not exist. Creating a new experiment.


<Experiment: artifact_location='sds://onelakecentralindia.pbidedicated.windows.net/c48bbed2-cdef-4f1e-9ff6-04461745e6d4/f5c90913-909c-4caa-a36d-d7e255ec8abd', creation_time=1780877702469, experiment_id='f5c90913-909c-4caa-a36d-d7e255ec8abd', last_update_time=1780877702469, lifecycle_stage='active', name='Fabric_MLflow_Demo', tags={}>

In [6]:
with mlflow.start_run():

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    mlflow.log_metric("accuracy", accuracy)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)

    print(f"Accuracy: {accuracy}")

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 9, Finished, Available, Finished, False)

2026/06/08 00:15:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils."


Accuracy: 1.0


In [9]:
with mlflow.start_run():

    model.fit(X_train, y_train)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model"
    )

    run_id = mlflow.active_run().info.run_id


StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 12, Finished, Available, Finished, False)

In [10]:
model_uri = f"runs:/{run_id}/model"

mlflow.register_model(
    model_uri=model_uri,
    name="Iris_Classifier"
)

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 13, Finished, Available, Finished, False)

Successfully registered model 'Iris_Classifier'.
2026/06/08 00:21:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Iris_Classifier, version 1
Created version '1' of model 'Iris_Classifier'.


<ModelVersion: aliases=[], creation_timestamp=1780878062980, current_stage='None', description='', last_updated_timestamp=1780878062980, name='Iris_Classifier', run_id='dd23f87e-4797-4d38-ad12-a6303d2da044', run_link='', source='abfss://c48bbed2-cdef-4f1e-9ff6-04461745e6d4@centralindia-onelake.dfs.core.windows.net/456f0e78-0e8b-48b9-9bf6-24d72d9d4904/Data/ea353cac-5c7c-4462-bf8d-fd0f7d187fcf/artifacts', status='READY', status_message='', tags={'synapseml.user.id': '454d3566-bf57-43ed-9831-6d978059976c',
 'synapseml.user.name': 'LabsKraft MentorITC'}, user_id='454d3566-bf57-43ed-9831-6d978059976c', version='1'>

In [11]:
for depth in [2, 4, 6, 8]:

    with mlflow.start_run():

        model = RandomForestClassifier(
            max_depth=depth,
            random_state=42
        )

        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        acc = accuracy_score(y_test, pred)

        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("accuracy", acc)

        print(depth, acc)

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 14, Finished, Available, Finished, False)

2 1.0


4 1.0


6 1.0


8 1.0


In [12]:
logged_model = "models:/Iris_Classifier/latest"

loaded_model = mlflow.pyfunc.load_model(logged_model)

predictions = loaded_model.predict(X_test)

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 16, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 17, Finished, Available, Finished, False)

StatementMeta(, 58d28124-0f25-4acd-9d91-7e1e26bee969, 18, Finished, Available, Finished, False)